# 02 · Build and check the fine-tuning dataset (Steps A5, A6, A7)

1. **A5:** re-render the T1–T3 demos with masks and 3D grounding, write the split manifest, convert to LeRobot format.
2. **A6:** compare computed object centers with simulator ground truth on every replayed step.
3. **A7:** rotate the camera 20 degrees; the images must change while the robot-frame centers stay put.

A T4 is enough. The build takes about an hour and is resumable: finished demos are skipped on restart.

It runs on the Colab local disk, because HDF5 files don't work reliably on the Drive mount, and copies the results to `MyDrive/groundedvla/datasets/`.

**Already built on the laptop?** Upload `D:\groundedvla\datasets\replayed` and `D:\groundedvla\datasets\lerobot` to `MyDrive/groundedvla/datasets/`. The manifest ships in the repo, so you only need the check cells at the end.

In [ ]:
# ---- edit once ----
REPO_URL = "https://github.com/kaustubhadhe1206/GroundedVLA.git"   # public repo, no token needed
BRANCH = "main"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "/content/GroundedVLA"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
!git log -1 --format="commit %h  %s"

In [ ]:
!apt-get -qq install -y libegl1 libgl1 rsync > /dev/null
%cd /content/GroundedVLA
!git pull --ff-only && git log -1 --format="commit %h  %s"
!bash scripts/colab_install.sh requirements.txt

In [ ]:
from pathlib import Path
from groundedvla.utils.colab import setup_session
paths = setup_session()          # env vars (EGL, caches on local disk), Drive folders, LIBERO config
DRIVE = Path(paths["root"])
BUILD = Path("/content/local/gvla_build")
(BUILD / "datasets").mkdir(parents=True, exist_ok=True)

## A5 · Re-render the demos (restores earlier progress from Drive, saves it back when done)

In [ ]:
!rsync -a {DRIVE}/datasets/libero {BUILD}/datasets/
!mkdir -p {DRIVE}/datasets/replayed
!rsync -a {DRIVE}/datasets/replayed {BUILD}/datasets/
!GVLA_ROOT={BUILD} python -u -m groundedvla.data.replay_demos --tasks T1 T2 T3
!rsync -a {BUILD}/datasets/replayed {DRIVE}/datasets/

## A5 · Split manifest and LeRobot conversion

In [ ]:
!GVLA_ROOT={BUILD} python -m groundedvla.data.make_split_manifest --seed 0
!GVLA_ROOT={BUILD} python -u -m groundedvla.data.convert_to_lerobot --repo-id local/gvla_libero --overwrite
!mkdir -p {DRIVE}/datasets/lerobot
!rsync -a --delete {BUILD}/datasets/lerobot/gvla_libero {DRIVE}/datasets/lerobot/
!mkdir -p {DRIVE}/outputs && cp groundedvla/data/split_manifest.json {DRIVE}/outputs/

The manifest is written inside this runtime's clone. If it differs from the one in the repo, commit the copy saved in `MyDrive/groundedvla/outputs/`, so all three variants read the same split.

## A5 · Dataset check: value ranges, random frames, and a real SmolVLA loss on our data

In [ ]:
!GVLA_ROOT={BUILD} python -m groundedvla.data.check_dataset --repo-id local/gvla_libero --device cuda
from IPython.display import Image, display
display(Image(str(BUILD / "outputs/dataset_checks/random_frames.png")))   # agentview | wrist | mask, all upright and aligned

## A6 + A7 · Grounding accuracy and camera invariance

In [ ]:
!GVLA_ROOT={BUILD} python -m groundedvla.grounding.check_grounding --dataset --live --tasks T1 T2 T3
for f in sorted((BUILD / "outputs/grounding_checks").glob("*.png")):
    print(f.name); display(Image(str(f), width=384))
!mkdir -p {DRIVE}/outputs && rsync -a {BUILD}/outputs/grounding_checks {BUILD}/outputs/dataset_checks {DRIVE}/outputs/